# Phase 9 — Threshold Analysis

## Objective

Determine an appropriate classification threshold for identifying high-risk customers.

The default classification threshold is 0.5, but this may not be suitable for an imbalanced credit-risk dataset.

We will:

1. Generate out-of-fold predicted probabilities using development data.
2. Evaluate multiple classification thresholds.
3. Compare precision, recall, F1-score and business implications.
4. Select a suitable threshold without using the locked test set.


In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\HP\RiskLens


In [17]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score

from src.config import RANDOM_SEED

In [18]:
print("Random seed:", RANDOM_SEED)
print("Import successful!")

Random seed: 42
Import successful!


In [19]:
from sklearn.calibration import CalibratedClassifierCV

In [20]:
from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV

In [34]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss,
    average_precision_score
)

from xgboost import XGBClassifier

from src.preprocessing import make_pipeline as project_make_pipeline
from src.config import RANDOM_SEED, TARGET_COL

print("Imports successful!")
print("Project root:", PROJECT_ROOT)

Imports successful!
Project root: c:\Users\HP\RiskLens


In [35]:
dev = pd.read_csv(PROJECT_ROOT / "data" / "dev.csv")

y_dev = dev[TARGET_COL]
X_dev = dev.drop(columns=[TARGET_COL])

print("Development features:", X_dev.shape)
print("Development target:", y_dev.shape)
print("Default rate:", y_dev.mean())

Development features: (120000, 10)
Development target: (120000,)
Default rate: 0.06684166666666666


In [36]:
spw = (y_dev == 0).sum() / (y_dev == 1).sum()

print("Scale positive weight:", spw)

Scale positive weight: 13.960728088766986


In [37]:
xgb_pipe = project_make_pipeline(
    XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=1.0,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=spw,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
)

print("Tuned XGBoost pipeline created!")

Tuned XGBoost pipeline created!


In [38]:
final_model = CalibratedClassifierCV(
    estimator=xgb_pipe,
    method="isotonic",
    cv=5
)

print("Calibrated XGBoost candidate created!")

Calibrated XGBoost candidate created!


In [39]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED
)

oof_probabilities = cross_val_predict(
    final_model,
    X_dev,
    y_dev,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

print("OOF predictions generated:", len(oof_probabilities))
print("Minimum probability:", oof_probabilities.min())
print("Maximum probability:", oof_probabilities.max())

OOF predictions generated: 120000
Minimum probability: 0.0
Maximum probability: 1.0


In [40]:
assert len(oof_probabilities) == len(y_dev)
assert np.isfinite(oof_probabilities).all()
assert ((oof_probabilities >= 0) & (oof_probabilities <= 1)).all()

print("All OOF prediction checks passed!")

All OOF prediction checks passed!


In [41]:
oof_results = pd.DataFrame({
    "actual_default": y_dev.to_numpy(),
    "predicted_probability": oof_probabilities
})

results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)

oof_results.to_csv(
    results_dir / "dev_oof_predictions.csv",
    index=False
)

print("Saved: results/dev_oof_predictions.csv")

Saved: results/dev_oof_predictions.csv


## Step 9.3 — Threshold Sweep

Evaluate classification performance across thresholds from 0.00 to 1.00 using the out-of-fold predictions generated in Step 9.2.

In [42]:
import json
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import confusion_matrix

# Project paths
PROJECT_ROOT = Path.cwd().parent

# Load OOF predictions
oof_results = pd.read_csv(
    PROJECT_ROOT / "results" / "dev_oof_predictions.csv"
)

y_true_oof = oof_results["actual_default"].to_numpy()
y_prob_oof = oof_results["predicted_probability"].to_numpy()

print("Actual labels:", len(y_true_oof))
print("Predicted probabilities:", len(y_prob_oof))
print("OOF predictions loaded successfully!")

Actual labels: 120000
Predicted probabilities: 120000
OOF predictions loaded successfully!


In [43]:
rows = []

thresholds = np.linspace(0, 1, 101)

for t in thresholds:

    yhat = (y_prob_oof >= t).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true_oof,
        yhat,
        labels=[0, 1]
    ).ravel()

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    rows.append({
        "threshold": round(float(t), 2),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "flag_rate": float((tp + fp) / len(y_true_oof)),
        "illustrative_cost": int(fn * 50000 + fp * 5000)
    })

threshold_df = pd.DataFrame(rows)

print("Threshold sweep completed!")
print("Total thresholds evaluated:", len(threshold_df))

threshold_df.head()

Threshold sweep completed!
Total thresholds evaluated: 101


,threshold,tp,fp,fn,tn,precision,recall,f1_score,flag_rate,illustrative_cost
0,0.00,8021,111979,0,0,0.066842,1.000000,0.125308,1.000000,559895000
1,0.01,7797,77404,224,34575,0.091513,0.972073,0.167278,0.710008,398220000
2,0.02,7461,54365,560,57614,0.120677,0.930183,0.213638,0.515217,299825000
3,0.03,7200,43026,821,68953,0.143352,0.897644,0.247223,0.418550,256180000
4,0.04,6944,36054,1077,75925,0.161496,0.865727,0.272212,0.358317,234120000


In [44]:
website_data_dir = PROJECT_ROOT / "website" / "data"

website_data_dir.mkdir(parents=True, exist_ok=True)

output_path = website_data_dir / "threshold_metrics.json"

with open(output_path, "w") as f:
    json.dump(rows, f, indent=4)

print("Threshold metrics saved successfully!")
print("Saved at:", output_path)

Threshold metrics saved successfully!
Saved at: c:\Users\HP\RiskLens\website\data\threshold_metrics.json


In [45]:
print("Number of threshold rows:", len(threshold_df))

print("\nBest threshold by F1-score:")
print(
    threshold_df.loc[
        threshold_df["f1_score"].idxmax()
    ]
)

print("\nLowest illustrative cost:")
print(
    threshold_df.loc[
        threshold_df["illustrative_cost"].idxmin()
    ]
)

print("\nJSON file exists:", output_path.exists())

Number of threshold rows: 101

Best threshold by F1-score:
threshold            2.300000e-01
tp                   3.947000e+03
fp                   5.706000e+03
fn                   4.074000e+03
tn                   1.062730e+05
precision            4.088884e-01
recall               4.920833e-01
f1_score             4.466448e-01
flag_rate            8.044167e-02
illustrative_cost    2.322300e+08
Name: 23, dtype: float64

Lowest illustrative cost:
threshold            9.000000e-02
tp                   5.696000e+03
fp                   1.679700e+04
fn                   2.325000e+03
tn                   9.518200e+04
precision            2.532343e-01
recall               7.101359e-01
f1_score             3.733368e-01
flag_rate            1.874417e-01
illustrative_cost    2.002350e+08
Name: 9, dtype: float64

JSON file exists: True


In [46]:
assert len(y_true_oof) == len(y_prob_oof)
assert len(y_true_oof) == len(dev)
assert np.isfinite(y_prob_oof).all()
assert ((y_prob_oof >= 0) & (y_prob_oof <= 1)).all()

print("OOF prediction checks passed!")

OOF prediction checks passed!


## Step 9.4 — Export Curve Data for Website

Export ROC, Precision–Recall and calibration curve points, along with summary metrics, as JSON files for website visualization.

All calculations use out-of-fold predictions from the development dataset.

In [47]:
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    brier_score_loss
)

from sklearn.calibration import calibration_curve
import json

In [48]:
fpr, tpr, roc_thresholds = roc_curve(
    y_true_oof,
    y_prob_oof
)

roc_data = [
    {
        "fpr": float(x),
        "tpr": float(y),
        "threshold": float(t)
    }
    for x, y, t in zip(fpr, tpr, roc_thresholds)
]

with open(website_data_dir / "roc_curve.json", "w") as f:
    json.dump(roc_data, f, indent=4)

print("ROC curve data saved!")

ROC curve data saved!


In [49]:
precision, recall, pr_thresholds = precision_recall_curve(
    y_true_oof,
    y_prob_oof
)

pr_data = [
    {
        "precision": float(p),
        "recall": float(r)
    }
    for p, r in zip(precision, recall)
]

with open(website_data_dir / "pr_curve.json", "w") as f:
    json.dump(pr_data, f, indent=4)

print("PR curve data saved!")

PR curve data saved!


In [50]:
fraction_positive, mean_predicted = calibration_curve(
    y_true_oof,
    y_prob_oof,
    n_bins=10,
    strategy="quantile"
)

calibration_data = [
    {
        "mean_predicted_probability": float(pred),
        "fraction_of_positives": float(actual)
    }
    for pred, actual in zip(mean_predicted, fraction_positive)
]

with open(website_data_dir / "calibration_curve.json", "w") as f:
    json.dump(calibration_data, f, indent=4)

print("Calibration curve data saved!")

Calibration curve data saved!


In [51]:
summary_metrics = {
    "roc_auc": float(roc_auc_score(y_true_oof, y_prob_oof)),
    "pr_auc": float(average_precision_score(y_true_oof, y_prob_oof)),
    "brier_score": float(brier_score_loss(y_true_oof, y_prob_oof)),
    "default_rate": float(np.mean(y_true_oof)),
    "total_customers": int(len(y_true_oof))
}

with open(website_data_dir / "summary_metrics.json", "w") as f:
    json.dump(summary_metrics, f, indent=4)

print(summary_metrics)

{'roc_auc': 0.8644573386140104, 'pr_auc': 0.397898170424963, 'brier_score': 0.04906364203971998, 'default_rate': 0.06684166666666666, 'total_customers': 120000}


In [52]:
required_files = [
    "threshold_metrics.json",
    "roc_curve.json",
    "pr_curve.json",
    "calibration_curve.json",
    "summary_metrics.json"
]

for filename in required_files:
    path = website_data_dir / filename
    print(filename, ":", path.exists())

threshold_metrics.json : True
roc_curve.json : True
pr_curve.json : True
calibration_curve.json : True
summary_metrics.json : True


## Phase 9 — Final Checkpoint

- [x] Threshold analysis notebook created.
- [x] Five-fold out-of-fold predictions generated using development data.
- [x] Prediction validation checks passed.
- [x] 101 classification thresholds evaluated.
- [x] Precision, recall, F1-score and flag rate calculated.
- [x] Illustrative business costs calculated.
- [x] Threshold metrics exported to JSON.
- [x] ROC curve data exported.
- [x] Precision–Recall curve data exported.
- [x] Calibration curve data exported.
- [x] Summary metrics exported.
- [x] Offline threshold slider implemented and tested.

**Threshold analysis summary:**

The best F1-score was achieved at threshold 0.23.
The lowest illustrative cost was achieved at threshold 0.09.

The two thresholds represent different precision-recall trade-offs.
Final threshold selection must consider business requirements.

All threshold analysis used development-set out-of-fold predictions.
The locked test set remains untouched.
